# 01 — Pregătire Dataset Detecție

Acest notebook acoperă întreg fluxul de pregătire a datelor pentru **Stage 1 (detecție)**:

1. Raport progres adnotare
2. Split dataset detecție (train / val / test)
3. Validare structură YOLO
4. Statistici dataset YOLO
5. Export crops pentru clasificare (Stage 2)
6. Merge dataset-uri YOLO (opțional — TACO + parks)

> **Rulează celulele de sus în jos.** Configurează parametrii din celula *Config* înainte de a continua.

In [9]:
# ── Imports comune ─────────────────────────────────────────────────────────────
import os
import random
import shutil
import sys
import csv
from collections import Counter, defaultdict
from pathlib import Path

import yaml
import cv2

# Asigurăm că rădăcina proiectului e în sys.path
REPO_ROOT = Path("../..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Rădăcina proiectului: {REPO_ROOT}")

Rădăcina proiectului: D:\TrashDetectionSystem


In [10]:
# ── CONFIG — modifică valorile de mai jos după necesitate ──────────────────────

# Directoare principale
DATASET_ROOT       = REPO_ROOT / "datasets" / "parks_detect"
IMAGES_ALL_DIR     = DATASET_ROOT / "images" / "all"
LABELS_ALL_DIR     = DATASET_ROOT / "labels" / "all"
DATASET_YAML       = DATASET_ROOT / "parks_detect.yaml"

# Parametri split
VAL_RATIO          = 0.15
TEST_RATIO         = 0.15
SEED               = 42
GROUP_BY           = "image"    # "image" = fiecare imagine e grup separat (recomandat pt dataset mic)
CLEAR_EXISTING     = True       # șterge split-urile vechi înainte de re-generare
ALLOW_EMPTY_LABELS = True       # True = imagini fără obiecte sunt negative valide
COPY_FILES         = False      # False = hardlink (mai rapid), True = copie

# Export crops
CROPS_IMAGES_DIR  = DATASET_ROOT / "images" / "train"   # sau images/all
CROPS_LABELS_DIR  = DATASET_ROOT / "labels" / "train"
CROPS_OUT_DIR     = REPO_ROOT / "datasets" / "parks_cls_unsorted" / "all"
CROPS_MANIFEST    = REPO_ROOT / "datasets" / "parks_cls_unsorted" / "crops_manifest.csv"
CROPS_MARGIN      = 0.05
CROPS_SKIP_EMPTY  = True

# Extensii imagini
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

print("Config încărcat OK")

Config încărcat OK


---
## 1. Raport Progres Adnotare
Verifică câte imagini au label, câte au label gol (negative) și câte nu au deloc label.

In [11]:
def image_files(directory: Path):
    if not directory.exists():
        return []
    return sorted(p for p in directory.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS)


def report_annotation_progress(images_dir: Path, labels_dir: Path):
    if not images_dir.exists():
        print(f"[EROARE] Directorul cu imagini nu există: {images_dir}")
        return

    images = image_files(images_dir)
    total = len(images)
    labeled = empty_lbl = missing = 0

    for img in images:
        lbl = labels_dir / f"{img.stem}.txt"
        if not lbl.exists():
            missing += 1
        elif lbl.read_text(encoding="utf-8").strip():
            labeled += 1
        else:
            empty_lbl += 1

    done = labeled + empty_lbl
    pct = done / total * 100 if total else 0.0

    print(f"Directorul imagini : {images_dir}")
    print(f"Directorul labels  : {labels_dir}")
    print(f"Total imagini      : {total}")
    print(f"  ✔ Cu obiecte     : {labeled}")
    print(f"  ○ Label gol      : {empty_lbl}")
    print(f"  ✗ Lipsă label    : {missing}")
    print(f"Progres adnotare   : {pct:.1f}%")

    if missing:
        print("\nImagini fără label (primele 10):")
        cnt = 0
        for img in images:
            if not (labels_dir / f"{img.stem}.txt").exists():
                print(f"  - {img.name}")
                cnt += 1
                if cnt >= 10:
                    break


report_annotation_progress(IMAGES_ALL_DIR, LABELS_ALL_DIR)

Directorul imagini : D:\TrashDetectionSystem\datasets\parks_detect\images\all
Directorul labels  : D:\TrashDetectionSystem\datasets\parks_detect\labels\all
Total imagini      : 48
  ✔ Cu obiecte     : 14
  ○ Label gol      : 34
  ✗ Lipsă label    : 0
Progres adnotare   : 100.0%


---
## 2. Split Dataset Detecție  
Împarte imaginile din `images/all` și `labels/all` în `train / val / test`.
Gruparea după prefix previne ca frame-uri din același video să ajungă în split-uri diferite (data leakage).

In [12]:
def group_key_for(stem: str, mode: str):
    if mode == "image":
        return stem
    marker = "_frame_"
    if marker in stem:
        return stem.split(marker, 1)[0]
    return stem.rsplit("_", 1)[0] if "_" in stem else stem


def assign_groups(group_keys, val_ratio, test_ratio, seed):
    shuffled = list(group_keys)
    random.Random(seed).shuffle(shuffled)
    total = len(shuffled)
    test_count = int(round(total * test_ratio))
    val_count  = int(round(total * val_ratio))
    if total >= 3:
        if test_ratio > 0 and test_count == 0: test_count = 1
        if val_ratio  > 0 and val_count  == 0: val_count  = 1
        overflow = test_count + val_count - total + 1
        while overflow > 0 and test_count > 0: test_count -= 1; overflow -= 1
        while overflow > 0 and val_count  > 0: val_count  -= 1; overflow -= 1
    split_map = {}
    for i, key in enumerate(shuffled):
        split_map[key] = "test" if i < test_count else ("val" if i < test_count + val_count else "train")
    return split_map


def link_or_copy(src: Path, dst: Path, copy_files: bool):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists(): dst.unlink()
    if copy_files:
        shutil.copy2(src, dst)
        return
    try:
        os.link(src, dst)
    except OSError:
        shutil.copy2(src, dst)


def split_detection_dataset(
    dataset_root, source_images, source_labels,
    val_ratio, test_ratio, seed, group_by, copy_files, clear_existing, allow_empty
):
    if not source_images.exists():
        print(f"[EROARE] Nu există: {source_images}"); return

    images = image_files(source_images)
    if not images:
        print(f"[EROARE] Nicio imagine în {source_images}"); return

    grouped = defaultdict(list)
    skipped = []
    for img in images:
        lbl = source_labels / f"{img.stem}.txt"
        if not lbl.exists() and not allow_empty:
            skipped.append(img); continue
        grouped[group_key_for(img.stem, group_by)].append((img, lbl))

    if not grouped:
        print("[EROARE] Nicio imagine eligibilă. Adaugă label-uri sau activează ALLOW_EMPTY_LABELS."); return

    if clear_existing:
        for split in ("train", "val", "test"):
            for folder in ("images", "labels"):
                d = dataset_root / folder / split
                if d.exists(): shutil.rmtree(d)

    for split in ("train", "val", "test"):
        for folder in ("images", "labels"):
            (dataset_root / folder / split).mkdir(parents=True, exist_ok=True)

    split_map = assign_groups(grouped.keys(), val_ratio, test_ratio, seed)
    stats = {"train": 0, "val": 0, "test": 0}

    for key, items in grouped.items():
        sp = split_map[key]
        for img, lbl in items:
            link_or_copy(img, dataset_root / "images" / sp / img.name, copy_files)
            dst_lbl = dataset_root / "labels" / sp / f"{img.stem}.txt"
            if lbl.exists(): link_or_copy(lbl, dst_lbl, copy_files)
            else: dst_lbl.write_text("", encoding="utf-8")
            stats[sp] += 1

    print(f"Grupe: {len(grouped)}  |  Train: {stats['train']}  Val: {stats['val']}  Test: {stats['test']}")
    if skipped:
        print(f"[WARN] {len(skipped)} imagini omise (lipsă label)")
    print("Split generat cu succes!")


split_detection_dataset(
    DATASET_ROOT, IMAGES_ALL_DIR, LABELS_ALL_DIR,
    VAL_RATIO, TEST_RATIO, SEED, GROUP_BY,
    COPY_FILES, CLEAR_EXISTING, ALLOW_EMPTY_LABELS
)

Grupe: 48  |  Train: 34  Val: 7  Test: 7
Split generat cu succes!


---
## 3. Validare Dataset YOLO
Verifică că structura de fișiere este corectă și că toate label-urile sunt valide (format YOLO, class_id în range, coordonate în [0,1]).

In [13]:
def load_dataset_config(yaml_path: Path):
    with yaml_path.open("r", encoding="utf-8") as f:
        data = yaml.safe_load(f)
    missing = sorted({"path", "train", "val", "names"} - set(data))
    if missing: raise ValueError(f"YAML lipsesc cheile: {missing}")
    return data


def resolve_root(yaml_path: Path, raw_path: str):
    p = Path(raw_path)
    if p.is_absolute(): return p
    for candidate in [p, Path.cwd() / p, yaml_path.parent / p]:
        if candidate.exists(): return candidate.resolve()
    return (Path.cwd() / p).resolve()


def validate_label_file(label_path: Path, class_count: int):
    problems = []
    for lineno, line in enumerate(label_path.read_text(encoding="utf-8").splitlines(), 1):
        if not line.strip(): continue
        parts = line.split()
        if len(parts) != 5:
            problems.append(f"{label_path.name}:{lineno} — trebuie 5 valori"); continue
        try:
            cls = int(parts[0]); coords = [float(v) for v in parts[1:]]
        except ValueError:
            problems.append(f"{label_path.name}:{lineno} — valori non-numerice"); continue
        if not (0 <= cls < class_count):
            problems.append(f"{label_path.name}:{lineno} — class_id {cls} invalid")
        xc, yc, w, h = coords
        if not (0 <= xc <= 1 and 0 <= yc <= 1):
            problems.append(f"{label_path.name}:{lineno} — centru în afara [0,1]")
        if not (0 < w <= 1 and 0 < h <= 1):
            problems.append(f"{label_path.name}:{lineno} — w/h invalid")
    return problems


def validate_yolo_dataset(yaml_path: Path):
    if not yaml_path.exists():
        print(f"[EROARE] YAML nu există: {yaml_path}"); return

    data = load_dataset_config(yaml_path)
    root = resolve_root(yaml_path, data["path"])
    names = data["names"]
    nc = len(names)

    all_problems = []
    total_images = 0

    print(f"YAML    : {yaml_path}")
    print(f"Root    : {root}")
    print(f"Clase   : {names}")

    for split in ("train", "val", "test"):
        split_val = data.get(split)
        if not split_val: continue
        img_dir = root / Path(split_val)
        lbl_dir = root / "labels" / Path(split_val).name
        imgs = image_files(img_dir)
        total_images += len(imgs)

        if not img_dir.exists():
            all_problems.append(f"Lipsă director imagini [{split}]: {img_dir}"); continue
        if not lbl_dir.exists():
            all_problems.append(f"Lipsă director labels [{split}]: {lbl_dir}"); continue

        for img in imgs:
            lbl = lbl_dir / f"{img.stem}.txt"
            if not lbl.exists():
                all_problems.append(f"[{split}] Lipsă label: {img.name}")
            else:
                all_problems.extend(validate_label_file(lbl, nc))

    print(f"Total imagini indexate: {total_images}")
    if all_problems:
        print(f"\n[FAIL] {len(all_problems)} probleme găsite:")
        for p in all_problems[:20]:
            print(f"  - {p}")
        if len(all_problems) > 20:
            print(f"  ... și încă {len(all_problems)-20}")
    else:
        print("\n[OK] Structura dataset-ului este validă pentru antrenare YOLO!")


validate_yolo_dataset(DATASET_YAML)

YAML    : D:\TrashDetectionSystem\datasets\parks_detect\parks_detect.yaml
Root    : D:\TrashDetectionSystem\datasets\parks_detect
Clase   : {0: 'trash'}
Total imagini indexate: 48

[OK] Structura dataset-ului este validă pentru antrenare YOLO!


---
## 4. Statistici Dataset YOLO
Afișează numărul de imagini, obiecte și distribuția pe clase pentru fiecare split.

In [14]:
def count_labels(label_path: Path):
    counts = Counter()
    if not label_path.exists(): return 0, counts
    for line in label_path.read_text(encoding="utf-8").splitlines():
        if line.strip():
            try: counts[int(line.split()[0])] += 1
            except (ValueError, IndexError): pass
    return sum(counts.values()), counts


def report_yolo_stats(yaml_path: Path):
    if not yaml_path.exists():
        print(f"[EROARE] YAML nu există: {yaml_path}"); return

    with yaml_path.open("r", encoding="utf-8") as f:
        data = yaml.safe_load(f)

    root = resolve_root(yaml_path, data["path"])
    raw_names = data["names"]
    names = {i: n for i, n in enumerate(raw_names)} if isinstance(raw_names, list) else {int(k): v for k, v in raw_names.items()}

    print(f"Dataset : {yaml_path.stem}")
    print(f"Clase   : {list(names.values())}\n")
    print(f"{'Split':<8} {'Imagini':>8} {'Negat.':>8} {'Obiecte':>9} " + " ".join(f"{n:>8}" for n in names.values()))
    print("-" * (34 + 9 * len(names)))

    totals = defaultdict(int)
    for split in ("train", "val", "test"):
        sv = data.get(split)
        if not sv: continue
        img_dir = root / Path(sv)
        lbl_dir = root / "labels" / Path(sv).name
        imgs = image_files(img_dir)
        empty = objs = 0
        class_counts = Counter()
        for img in imgs:
            n, cc = count_labels(lbl_dir / f"{img.stem}.txt")
            if n == 0: empty += 1
            objs += n; class_counts.update(cc)
        row = f"{split:<8} {len(imgs):>8} {empty:>8} {objs:>9} " + " ".join(f"{class_counts.get(i,0):>8}" for i in names)
        print(row)
        totals["imgs"] += len(imgs); totals["empty"] += empty; totals["objs"] += objs
        for i in names: totals[f"cls_{i}"] += class_counts.get(i, 0)

    print("-" * (34 + 9 * len(names)))
    row = f"{'TOTAL':<8} {totals['imgs']:>8} {totals['empty']:>8} {totals['objs']:>9} " + " ".join(f"{totals.get(f'cls_{i}',0):>8}" for i in names)
    print(row)


report_yolo_stats(DATASET_YAML)

Dataset : parks_detect
Clase   : ['trash']

Split     Imagini   Negat.   Obiecte    trash
-------------------------------------------
train          34       26        10       10
val             7        5         4        4
test            7        3         7        7
-------------------------------------------
TOTAL          48       34        21       21


---
## 5. Export Crops pentru Clasificare
Decupează obiectele detectate din imagini pe baza label-urilor YOLO și le salvează ca imagini individuale pentru antrenarea clasificatorului (Stage 2).

In [15]:
def export_yolo_crops(images_dir, labels_dir, out_dir, manifest_path, margin=0.05, skip_empty=True):
    if not images_dir.exists():
        print(f"[EROARE] {images_dir}"); return
    if not labels_dir.exists():
        print(f"[EROARE] {labels_dir}"); return

    out_dir.mkdir(parents=True, exist_ok=True)
    manifest_path.parent.mkdir(parents=True, exist_ok=True)

    imgs = image_files(images_dir)
    written = skipped = 0

    with manifest_path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["crop_path","source_image","label_path","box_index","class_id","x1","y1","x2","y2"])

        for img_path in imgs:
            lbl_path = labels_dir / f"{img_path.stem}.txt"
            if not lbl_path.exists(): skipped += 1; continue
            lines = [l.strip() for l in lbl_path.read_text(encoding="utf-8").splitlines() if l.strip()]
            if not lines and skip_empty: skipped += 1; continue

            frame = cv2.imread(str(img_path))
            if frame is None: skipped += 1; continue
            h, w = frame.shape[:2]

            for idx, line in enumerate(lines):
                parts = line.split()
                if len(parts) != 5: continue
                cls_id = int(parts[0])
                xc, yc, bw, bh = map(float, parts[1:])
                x1 = (xc - bw/2) * w; y1 = (yc - bh/2) * h
                x2 = (xc + bw/2) * w; y2 = (yc + bh/2) * h
                pad_x = (x2-x1)*margin; pad_y = (y2-y1)*margin
                l = max(int(round(x1-pad_x)), 0); t = max(int(round(y1-pad_y)), 0)
                r = min(int(round(x2+pad_x)), w); b = min(int(round(y2+pad_y)), h)
                if r <= l or b <= t: continue
                crop = frame[t:b, l:r]
                crop_name = f"{img_path.stem}_crop_{idx:03d}{img_path.suffix}"
                crop_path = out_dir / crop_name
                if not cv2.imwrite(str(crop_path), crop): continue
                writer.writerow([crop_path.as_posix(), img_path.as_posix(), lbl_path.as_posix(), idx, cls_id, l, t, r, b])
                written += 1

    print(f"Crops scrise    : {written}")
    print(f"Imagini omise   : {skipped}")
    print(f"Output director : {out_dir}")
    print(f"Manifest CSV    : {manifest_path}")


export_yolo_crops(CROPS_IMAGES_DIR, CROPS_LABELS_DIR, CROPS_OUT_DIR, CROPS_MANIFEST, CROPS_MARGIN, CROPS_SKIP_EMPTY)

Crops scrise    : 10
Imagini omise   : 26
Output director : D:\TrashDetectionSystem\datasets\parks_cls_unsorted\all
Manifest CSV    : D:\TrashDetectionSystem\datasets\parks_cls_unsorted\crops_manifest.csv


---
## 6. Merge Dataset-uri YOLO (Opțional)
Combină mai multe dataset-uri YOLO (ex: TACO + parks) într-un singur dataset final. Fiecare sursă primește un prefix pentru a evita conflictele de nume.

In [1]:
# Config merge — ajustează după necesitate
MERGE_SOURCES = [
    (REPO_ROOT / "datasets" / "parks_detect",  "parks"),
    # (REPO_ROOT / "datasets" / "taco_yolo",   "taco"),   # decomentează dacă ai descărcat TACO
]
MERGE_OUTPUT = REPO_ROOT / "datasets" / "parks_detect_full"


def merge_yolo_datasets(sources_with_prefix, out_root: Path):
    for split in ("train", "val", "test"):
        for folder in ("images", "labels"):
            (out_root / folder / split).mkdir(parents=True, exist_ok=True)

    for src_root, prefix in sources_with_prefix:
        src_root = Path(src_root)
        if not src_root.exists():
            print(f"[WARN] Sursă inexistentă: {src_root}"); continue
        for split in ("train", "val", "test"):
            src_img = src_root / "images" / split
            src_lbl = src_root / "labels" / split
            if not src_img.exists(): continue
            copied = 0
            for img in src_img.iterdir():
                if not img.is_file() or img.suffix.lower() not in IMAGE_EXTS: continue
                new_stem = f"{prefix}_{img.stem}"
                shutil.copy2(img, out_root / "images" / split / f"{new_stem}{img.suffix}")
                src_l = src_lbl / f"{img.stem}.txt"
                dst_l = out_root / "labels" / split / f"{new_stem}.txt"
                if src_l.exists(): shutil.copy2(src_l, dst_l)
                else: dst_l.write_text("", encoding="utf-8")
                copied += 1
            print(f"  [{prefix}] {split}: {copied} imagini copiate")

    print(f"\nMerge complet → {out_root}")


# Decomentează pentru a rula merge-ul:
# merge_yolo_datasets(MERGE_SOURCES, MERGE_OUTPUT)

NameError: name 'REPO_ROOT' is not defined